# Welcome to NeuroMorph!

NeuroMorph is a deep learning pipeline that generates cortical segmentation and cortical reconstructions with high speed and accuracy.





Below is a schema of how NeuroMorph is organised

![NeuroMorph_Schema.png](./media/NeuroMorph_Schema.png)

| Input Parameter | Requirements |
|---|---|
|Modality	| T1-weighted MRI
|Format	NIfTI | (.nii, .nii.gz)
|Resolution	| 1mm isotropic
|Orientation |	LIA recommended; the pipeline will attempt to reorient automatically
|Preprocessing	| Internal preprocessing: shape, size and orientation conform only. No other preprocessing is required e.g., skull stripping, bias field correction, etc.|
|Space	| Any (No registration needed) | 

# Installation

## Docker / Singularity

**1. Build the Singularity image from Docker Hub**

Run the following command from the directory where you want the `.simg` file to be created.

```bash
cd /local/path/to/NeuroMorph/
singularity build ./deep_thickness_v0.12_release.simg docker://rocknroll87q/deep_thickness:v0.12_release
```

This pulls the public image `rocknroll87q/deep_thickness:v0.12_release` from Docker Hub and converts it into a Singularity image file (`.simg`) in the current directory.

Build times vary depending on network speed and system load, but typically take a few minutes.

**2. Verify the build**

Once the build completes, confirm the image was created successfully.

```bash
ls -lh deep_thickness_v0.12_release.simg
```



**3. Running the container**

Add the `--nv` flag so Singularity binds the host NVIDIA drivers into the container.

```bash
singularity shell --cleanenv --nv \
    --env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/.singularity.d/libs" \
    -B ./:/NeuroMorph/ \
    ./deep_thickness_v0.12_release.simg
cd /NeuroMorph/

```

If the singularity version is older than 3.6 then use

```bash
singularity shell --cleanenv --nv -B ./:/NeuroMorph/ ./deep_thickness_v0.12_release.simg
cd /NeuroMorph/
```

# Quickstart Inference

If you want to generate the outputs as quickly as possible with your own data, you can run the following command in the singularity container shell. Make sure to replace the input and output paths with your own.


```bash
# Launch the singularity container with input and output directories 
singularity shell --cleanenv --nv \
    --env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/.singularity.d/libs" \
    -B ./:/NeuroMorph/ \
    -B /local/path/to/input:/NeuroMorph/neuromorph_input \
    -B /local/path/to/output:/NeuroMorph/neuromorph_output \
    ./deep_thickness_v0.12_release.simg 
```

```bash 
# Generate outputs for all files (containing .nii or .nii.gz extension) in the input directory
python3 /NeuroMorph/scripts/run_inference.py \
         --data.vol_in='/NeuroMorph/neuromorph_input/' \
         --data.exp_path='/NeuroMorph/neuromorph_output' \
         --input_output.no_of_save_outputs=-1 
```

# Inference Examples

Follow the steps below to run different types of inference on real example MRI data from the AHEAD dataset

In [ ]:
from pathlib import Path
import sys
sys.path.append("/NeuroMorph/")

IN_DIR = Path('./tmp/neuromorph_input')
OUT_ROOT = Path('./tmp/neuromorph_output')


In [ ]:
# make input and output directories
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

## Generate Examples

In [ ]:
# install openneuro if not already installed
try:
    import openneuro
except ImportError:
    !pip install openneuro-py

In [ ]:
!{sys.executable} ./scripts/get_test_data.py --output {IN_DIR} --mode T1w -n 2

## Run Examples via Singularity


If you would like to run the below example cells via singularity, you can run the following command in your terminal which will create a local host. Otherwise, you can paste the python commands into a singularity shell and run them there. 

```bash
singularity shell --cleanenv --nv \
    --env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/.singularity.d/libs" \
    -B ./:/NeuroMorph/ \
    ./deep_thickness_v0.12_release.simg 
```

```bash
pip install --no-cache-dir notebook jupyterlab ipykernel
```

```bash
python -m notebook --no-browser --port=8888 --ip=0.0.0.0
```

## Single Volume Test

You can run the cells directly if porting the notebook via the singularity container

In [ ]:
import sys
!{sys.executable}  /NeuroMorph/scripts/run_inference.py --data.exp_path='{OUT_ROOT}' \
    --data.vol_in='{IN_DIR}/sub-0001_T1w.nii.gz' \
    --data.exp_path='{OUT_ROOT}'

Else copy and paste the run commands into singularity container shell

```bash
cd /NeuroMorph/
python3 /NeuroMorph/scripts/run_inference.py \
         --data.vol_in='/NeuroMorph/tmp/neuromorph_input/sub-0001_T1w.nii.gz' \
         --data.exp_path='/NeuroMorph/tmp/neuromorph_output' 
```

## Multiple Volume Test

To process an entire directory of volumes, we can simply set the `--data.vol_in` argument to the directory containing the volumes. The script will automatically find all `.nii.gz` files in that directory and process them one by one.


In [ ]:
!{sys.executable} ./scripts/run_inference.py \
         --data.vol_in='/NeuroMorph/tmp/neuromorph_input/' \
         --data.exp_path='/NeuroMorph/tmp/neuromorph_output' \
         --input_output.no_of_save_outputs=-1 

```bash
python3 /NeuroMorph/scripts/run_inference.py \
         --data.vol_in='/NeuroMorph/tmp/neuromorph_input/' \
         --data.exp_path='/NeuroMorph/tmp/neuromorph_output' \
         --input_output.no_of_save_outputs=-1 
```

Alternatively, we can set the parameter `--data.file_identifier` to a substring that can be used to find the T1w files when recursively searching a data.vol_in directory. For example, if we set `--data.file_identifier='T1w'`, the script will look for all files containing 'T1w' in their names.

## Inference From CSV

We can process a list of volumes specified in a CSV file. The CSV file should have a column named 'T1' that contains the paths to the input volumes. Paths need to be relative if using a singularity container e.g. "/NeuroMorph/data/". A "Subject" column is also needed in order to correctly identify each subject. 

In [ ]:
!{sys.executable} ./scripts/run_inference.py \
         --data.Path_in_csv='/NeuroMorph/tmp/neuromorph_input/' \
         --data.Filename_csv='subjects.csv' \
         --data.exp_path='/NeuroMorph/tmp/neuromorph_output' 

```bash
python3 /NeuroMorph/scripts/run_inference.py \
         --data.Path_in_csv='/NeuroMorph/tmp/neuromorph_input/' \
         --data.Filename_csv='subjects.csv' \
         --data.exp_path='/NeuroMorph/tmp/neuromorph_output'
```

# Saving Options

The quantitative morphological features for each subject will always be saved to a CSV file in the output directory. 

You can optionally save intermediate objects such as the cortical segementation and surfacce reconstruction using the following parameters:


Set any of the following to `True`/`False` to flag an output.

| Flag | Description | Notes |
|---|---|---|
| `--input_output.no_of_save_outputs` | Number of subjects to save | |
| `--input_output.out_T1`   | Conformed T1w volume used for models | |
| `--input_output.out_segmentation`   | Cortical segmentation volume | |
| `--input_output.out_surface_type`   | Output WM, Pial, or both surfaces | |
| `--input_output.out_surface_mesh`   | Cortical surface reconstruction (cortical thickness) | Saved as a `.ply` mesh with CTh overlay by default |
| `--input_output.out_curvature_mesh` | Cortical surface reconstruction (curvature) | Saved as a `.ply` mesh with curvature values overlay |
| `--input_output.out_t1w_overlay_mesh`   | Cortical surface reconstruction (T1w overlay) | Saved as a `.ply` mesh with interpolated T1w intensity values overlay |
| `--input_output.out_surface_fs` | Cortical surface reconstruction (FreeSurfer format) | Saves the selected surface reconstruction as a `.surf` file, with an additional `.thickness` file for loading into FreeSurfer |
| `--input_output.out_level_set` | ImplicitNet predicted level set | Saves the level set predicted by the ImplicitNet model |
| `--input_output.out_distance_set` | ImplicitNet predicted distance set | Saves the distance set predicted by the ImplicitNet model |

# Advanced Options

## Surface Reconstruction Calibration

The parameters below are recommended to remain at their defaults for best results. They are provided in case the user wishes to alter the final surface reconstruction. See the manuscript for further detail on any of these parameters.

#### Marching Cubes Level

`testing.pial_mc_level`
`testing.wm_mc_level`

These parameters alter the level of the marching cubes algorithm for each mesh type, effectively cutting at different depths into the tissue when generating the initial unprocessed mesh. Increasing the mc_level results in deeper sulci and smaller gyri, and vice versa. We suggest keeping the default parameters, which were rigorously optimised.

#### Normal Multiplier

`testing.pial_norm_multiplier`
`testing.wm_norm_multiplier`

Altering the norm_multiplier inflates the unprocessed surface, moving vertices along their face normals to retain cortical geometry. This is primarily used to recover the effects of mc_level, but can also be used for other purposes.

#### Mesh Decimation Target

`testing.mesh_decimation_target`

The default number of vertices is set to 7500. We found this to be the minimum number at which the cortex retains its geometry while reducing redundancy of vertices per face.

#### Mesh Post Processing

`testing.apply_mesh_post_processing`

To improve mesh quality and reduce the number of intersections, post processing can optionally be applied to the mesh. This is achieved by applying

* Minor Taubin smoothing to the whole mesh
* Laplacian smoothing on intersecting faces
* Up to 5 rounds of Taubin smoothing on intersecting faces

## Compute and Runtime Settings

#### Keep Mesh in Memory

`keep_mesh_in_memory`

When running inference or model testing, predicted meshes can be kept in memory after parallel processing if they are needed for further analysis, or removed from memory if they are only generated for metrics. Meshes are still saved to disk regardless of this setting.

#### Limit CPU Count

`limit_cpu_count`

Limits the number of CPU cores used during inference or model testing. If set to None, all available cores are used minus one.

#### Device

`device`

Specifies the device used for inference. Options are 'auto', 'cpu' or 'gpu'. If set to 'auto', the script uses the GPU when available, otherwise it falls back to the CPU.